# Vidal et al. (2019) Case 2: Full-to-Full Transfer

Case 2 transfers only the source Case 1 `model_state_dict` into a fresh target Vidal LSTM. The target optimizer, scheduler, and min-max scaler are recreated, all paper trainable parameters remain unlocked, and the target model trains on the target `full_train` split before evaluation on the target `test` split.

This notebook defaults to a quick smoke run only. It does not automatically launch formal 515-epoch experiments.

## 1. Runtime and repository setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import subprocess
import sys

REPO_ROOT = Path('/content/battery-soc-transfer-benchmark')
DRIVE_ROOT = Path('/content/drive/MyDrive/battery_soc_benchmark')
VIDAL_V2_ROOT = DRIVE_ROOT / 'data/labelled/vidal_sequence_v2'
CASE1_OUTPUT_ROOT = DRIVE_ROOT / 'outputs/vidal_case1'
CASE2_OUTPUT_ROOT = DRIVE_ROOT / 'outputs/vidal_case2'
LOCAL_CACHE_ROOT = Path('/content/battery_soc_cache/vidal_case2')

if not REPO_ROOT.exists():
    subprocess.run([
        'git', 'clone',
        'https://github.com/zhangcastle/battery-soc-transfer-benchmark.git',
        str(REPO_ROOT),
    ], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_ROOT)], check=True)

%cd /content/battery-soc-transfer-benchmark
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

print('Paths configured:')
print(REPO_ROOT)
print(VIDAL_V2_ROOT)
print(CASE1_OUTPUT_ROOT)
print(CASE2_OUTPUT_ROOT)

## 2. Git and environment preflight

Formal runs require a known commit and a clean working tree. If `dirty=True`, restart the runtime or restore tracked files before formal execution. Do not reset or checkout from this notebook.

In [ ]:
import torch
import pandas as pd
from IPython.display import Image, display

commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip()
dirty_text = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_ROOT, text=True).strip()
dirty = bool(dirty_text)
print('commit:', commit)
print('dirty:', dirty)
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
print('device_name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
if dirty:
    print('FORMAL SAFETY: restart runtime or restore tracked files before formal execution.')

## 3. SOC v2 input preflight

In [ ]:
from battery_benchmark.data.vidal_sequence_v2_dataset import validate_case1_roots

checks = validate_case1_roots(VIDAL_V2_ROOT)
display(checks)
assert (checks['status'] == 'PASS').all(), 'Required SOC v2 inputs are missing.'

## 4. Case 1 formal checkpoint discovery

Only formal Case 1 runs with `epoch_515.pt` and `metrics.json` are valid Case 2 sources.

In [ ]:
import torch

rows = []
for dataset in ['panasonic_18650pf', 'lg_hg2', 'samsung_30t']:
    root = CASE1_OUTPUT_ROOT / dataset / 'seed_42'
    for meta_path in sorted(root.glob('*/run_metadata.json')):
        run = meta_path.parent
        metadata = json.loads(meta_path.read_text())
        ckpt = run / 'checkpoints' / 'epoch_515.pt'
        metrics = run / 'metrics.json'
        epoch = None
        if ckpt.is_file():
            try:
                epoch = int(torch.load(ckpt, map_location='cpu', weights_only=False).get('epoch'))
            except Exception as exc:
                epoch = f'ERROR: {type(exc).__name__}'
        rows.append({
            'dataset': dataset,
            'run_path': str(run),
            'checkpoint_path': str(ckpt),
            'checkpoint_epoch': epoch,
            'run_mode': metadata.get('run_mode'),
            'development_only': metadata.get('development_only'),
            'checkpoint_exists': ckpt.is_file(),
            'metrics_exists': metrics.is_file(),
            'git_commit': metadata.get('git_commit'),
        })

case1_runs = pd.DataFrame(rows)
if case1_runs.empty:
    print('No Case 1 runs found under:', CASE1_OUTPUT_ROOT)
else:
    display(case1_runs)

formal = case1_runs[
    (case1_runs['run_mode'] == 'formal')
    & (case1_runs['development_only'] == False)
    & (case1_runs['checkpoint_exists'])
    & (case1_runs['metrics_exists'])
    & (case1_runs['checkpoint_epoch'] == 514)
].copy() if not case1_runs.empty else pd.DataFrame()

display(formal)
print('valid formal source checkpoints:', len(formal))

## 5. Default quick smoke: LG HG2 -> Samsung 30T

This is an integration smoke only: 2 epochs, at most 32 target training windows, batch size 32, output under `_smoke`. It is not a performance result.

In [ ]:
SOURCE = 'lg_hg2'
TARGET = 'samsung_30t'
match = formal[formal['dataset'] == SOURCE]
assert not match.empty, f'Missing formal Case 1 source checkpoint for {SOURCE}'
SOURCE_CHECKPOINT = Path(match.iloc[-1]['checkpoint_path'])
print('Using source checkpoint:')
print(SOURCE_CHECKPOINT)

!python scripts/run_vidal_case2.py   --source {SOURCE}   --target {TARGET}   --config configs/experiments/vidal_case2_smoke.yaml   --input-root {VIDAL_V2_ROOT}   --output-root {CASE2_OUTPUT_ROOT}   --case1-output-root {CASE1_OUTPUT_ROOT}   --source-checkpoint {SOURCE_CHECKPOINT}   --cache-root {LOCAL_CACHE_ROOT}   --run-mode smoke   --device auto   --max-train-windows 32

## 6. Inspect latest quick smoke

In [ ]:
pair_root = CASE2_OUTPUT_ROOT / '_smoke' / f'source_{SOURCE}__target_{TARGET}' / 'seed_42'
runs = sorted(pair_root.glob('*'), key=lambda p: p.stat().st_mtime)
assert runs, f'No quick smoke runs found under {pair_root}'
latest_smoke = runs[-1]
print('latest quick smoke:')
print(latest_smoke)
metadata = json.loads((latest_smoke / 'run_metadata.json').read_text())
metrics = json.loads((latest_smoke / 'metrics.json').read_text())
print('source checkpoint:', metadata['source_checkpoint']['source_checkpoint_path'])
print('source checkpoint epoch:', metadata['source_checkpoint']['source_checkpoint_epoch'])
print('actual_train_window_count:', metadata['actual_train_window_count'])
print('initial_mse_window_count:', metadata['initial_mse_window_count'])
print('initial target MSE:', metadata['initial_target_mse_before_training'])
print('final target train MSE:', metadata['final_target_train_mse'])
print('optimizer_reinitialized:', metadata['optimizer_reinitialized'])
print('scheduler_reinitialized:', metadata['scheduler_reinitialized'])
print('target_scaler_refit:', metadata['target_scaler_refit'])
print('device:', metadata['device'])
print('metrics:', metrics)
for figure in sorted((latest_smoke / 'figures').glob('*.png')):
    display(Image(filename=str(figure)))

## 7. Optional 5-epoch full-window diagnostic

This development-only diagnostic uses all target training windows for 5 epochs with batch size 512. It is useful for checking training stability, still writes under `_smoke`, and is not a formal run.

The command cell below is intentionally not executed by default.

In [ ]:
# Optional diagnostic. Run manually when you want a full-window development check.
# !python scripts/run_vidal_case2.py #   --source {SOURCE} #   --target {TARGET} #   --config configs/experiments/vidal_case2_diagnostic.yaml #   --input-root {VIDAL_V2_ROOT} #   --output-root {CASE2_OUTPUT_ROOT} #   --case1-output-root {CASE1_OUTPUT_ROOT} #   --source-checkpoint {SOURCE_CHECKPOINT} #   --cache-root {LOCAL_CACHE_ROOT} #   --run-mode smoke #   --device auto

## 8. Inspect latest diagnostic

In [ ]:
diagnostic_runs = []
pair_root = CASE2_OUTPUT_ROOT / '_smoke' / f'source_{SOURCE}__target_{TARGET}' / 'seed_42'
for run in sorted(pair_root.glob('*'), key=lambda p: p.stat().st_mtime):
    cfg_path = run / 'resolved_config.yaml'
    meta_path = run / 'run_metadata.json'
    if not cfg_path.is_file() or not meta_path.is_file():
        continue
    text = cfg_path.read_text()
    if 'epochs: 5' in text and 'max_train_windows: null' in text:
        diagnostic_runs.append(run)

if not diagnostic_runs:
    print('No 5-epoch full-window diagnostic run found yet.')
else:
    latest_diagnostic = diagnostic_runs[-1]
    print('latest diagnostic:')
    print(latest_diagnostic)
    metadata = json.loads((latest_diagnostic / 'run_metadata.json').read_text())
    metrics = json.loads((latest_diagnostic / 'metrics.json').read_text())
    print('initial target MSE:', metadata['initial_target_mse_before_training'])
    print('final target train MSE:', metadata['final_target_train_mse'])
    print('actual_train_window_count:', metadata['actual_train_window_count'])
    print('device:', metadata['device'])
    print('metrics:', metrics)

## 9. Formal single-pair command

This command is not executed automatically. It uses the checkpoint discovered above rather than a placeholder path.

In [ ]:
if dirty:
    print('Formal run blocked: restart runtime or restore tracked files before formal execution.')
elif not torch.cuda.is_available():
    print('CUDA is not available. Formal run is allowed by code but likely slow.')
elif formal[formal['dataset'] == SOURCE].empty:
    print(f'Missing formal source checkpoint for {SOURCE}.')
else:
    print('Formal single-pair command:')
    print(f"python scripts/run_vidal_case2.py --source {SOURCE} --target {TARGET} "
          f"--config configs/experiments/vidal_case2.yaml "
          f"--input-root {VIDAL_V2_ROOT} "
          f"--output-root {CASE2_OUTPUT_ROOT} "
          f"--case1-output-root {CASE1_OUTPUT_ROOT} "
          f"--source-checkpoint {SOURCE_CHECKPOINT} "
          "--run-mode formal --device auto")

## 10. Formal all-pairs command

This command is not executed automatically. It runs six directed pairs, each for 515 epochs, and requires all three formal Case 1 source checkpoints.

In [ ]:
required = {'panasonic_18650pf', 'lg_hg2', 'samsung_30t'}
found = set(formal['dataset']) if not formal.empty else set()
if dirty:
    print('Formal all-pairs blocked: restart runtime or restore tracked files before formal execution.')
elif not required <= found:
    print('Formal all-pairs blocked. Missing source checkpoints for:', sorted(required - found))
else:
    if not torch.cuda.is_available():
        print('CUDA is not available. Formal all-pairs will be slow.')
    print('Formal all-pairs command:')
    print(f"python scripts/run_vidal_case2.py --all-pairs "
          f"--config configs/experiments/vidal_case2.yaml "
          f"--input-root {VIDAL_V2_ROOT} "
          f"--output-root {CASE2_OUTPUT_ROOT} "
          f"--case1-output-root {CASE1_OUTPUT_ROOT} "
          f"--source-checkpoint-root {CASE1_OUTPUT_ROOT} "
          "--run-mode formal --device auto")

## 11. Result summary

In [ ]:
summary_path = CASE2_OUTPUT_ROOT / 'case2_summary' / 'case2_summary.csv'
if summary_path.is_file():
    display(pd.read_csv(summary_path))
else:
    print('No formal Case 2 summary found yet:', summary_path)